In [4]:
import numpy as np
import pandas as pd
import joblib

# 定义文件夹路径
folder_path_qf = "final_model/qf/rf"
folder_path_kl = "final_model/kl/rf"

# 权重定义
weights_qf = {'rf': 0.2}
weights_kl = {'rf': 0.2}

# 读取数据
df_qf = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name='qf_21')
df_kl = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name='kl_21')
display(df_qf, df_kl)
print(df_qf.shape)
# 加载测试集
X_test_qf = df_qf.iloc[:, :-2]
y_test_qf = df_qf['Yield_Strength']

X_test_kl = df_kl.iloc[:, :-2]
y_test_kl = df_kl['Tensile_Strength （UTS）']

# 选择所有样本
sample_indices_qf = np.arange(X_test_qf.shape[0])
sample_indices_kl = np.arange(X_test_kl.shape[0])

# **特征定义**
feature_ranges = {
    'Fraction of basal phase': np.arange(0, 21, 0.5),
    'Fraction of prismatic phase': np.arange(0, 21, 0.5)
}

# 保存屈服强度分析结果
with pd.ExcelWriter('AAA_qf_multiple_single_variable_analysis_fraction.xlsx', engine='openpyxl') as writer:
    for feature in feature_ranges.keys():
        all_results = []  # 用于存储所有样本的该特征预测结果
        ranges = feature_ranges[feature]

        for sample_index in sample_indices_qf:
            sample = X_test_qf.iloc[sample_index].copy()
            predictions_changes = []

            for value in ranges:
                modified_sample = sample.copy()
                modified_sample[feature] = value
                modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_qf.columns)

                prediction_for_this_value = np.zeros(1)
                for model_name, weight in weights_qf.items():
                    for i in range(0, 5):
                        model_path = f'{folder_path_qf}/{model_name}_{i}.pkl'
                        model = joblib.load(model_path)
                        prediction_for_this_value += model.predict(modified_sample_df) * weight

                predictions_changes.append((sample_index, value, prediction_for_this_value[0]))

            # 计算该样本在该特征下的预测范围
            pred_min = min([x[2] for x in predictions_changes])
            pred_max = max([x[2] for x in predictions_changes])
            pred_range = pred_max - pred_min

            print(f"🔸 **样本 {sample_index} 在 {feature} 下的预测范围**: {pred_min:.2f} ~ {pred_max:.2f} (差值 = {pred_range:.2f})")

            if pred_range > 30:
                print(f"⚠️ **警告：样本 {sample_index} 在 {feature} 上的预测范围超过 30** ⚠️")

            all_results.extend(predictions_changes)

        # 保存所有样本的该特征预测结果
        results_df = pd.DataFrame(all_results, columns=['sample_index', 'feature_value', 'Predicted Yield Strength'])
        results_df.to_excel(writer, sheet_name=feature.replace(" ", "_"), index=False)

print("\n✅ 屈服强度单变量分析完成，结果已保存到 AAA_qf_multiple_single_variable_analysis_fraction.xlsx")

# 保存抗拉强度分析结果
with pd.ExcelWriter('AAA_kl_multiple_single_variable_analysis_fraction.xlsx', engine='openpyxl') as writer:
    for feature in feature_ranges.keys():
        all_results = []  # 用于存储所有样本的该特征预测结果
        ranges = feature_ranges[feature]

        for sample_index in sample_indices_kl:
            sample = X_test_kl.iloc[sample_index].copy()
            predictions_changes = []

            for value in ranges:
                modified_sample = sample.copy()
                modified_sample[feature] = value
                modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_kl.columns)

                prediction_for_this_value = np.zeros(1)
                for model_name, weight in weights_kl.items():
                    for i in range(0, 5):
                        model_path = f'{folder_path_kl}/{model_name}_{i}.pkl'
                        model = joblib.load(model_path)
                        prediction_for_this_value += model.predict(modified_sample_df) * weight

                predictions_changes.append((sample_index, value, prediction_for_this_value[0]))

            # 计算该样本在该特征下的预测范围
            pred_min = min([x[2] for x in predictions_changes])
            pred_max = max([x[2] for x in predictions_changes])
            pred_range = pred_max - pred_min

            print(f"🔸 **样本 {sample_index} 在 {feature} 下的预测范围**: {pred_min:.2f} ~ {pred_max:.2f} (差值 = {pred_range:.2f})")

            if pred_range > 30:
                print(f"⚠️ **警告：样本 {sample_index} 在 {feature} 上的预测范围超过 30** ⚠️")

            all_results.extend(predictions_changes)

        # 保存所有样本的该特征预测结果
        results_df = pd.DataFrame(all_results, columns=['sample_index', 'feature_value', 'Predicted Tensile Strength'])
        results_df.to_excel(writer, sheet_name=feature.replace(" ", "_"), index=False)

print("\n✅ 抗拉强度单变量分析完成，结果已保存到 AAA_kl_multiple_single_variable_analysis_fraction.xlsx")


,Length of prismatic phase,mean AtomicRadius,Zr fraction,Fraction of prismatic phase,Width of prismatic phase,frac f valence electrons,Diameter of prismatic phase,Diameter of basal phase,Width of basal phase,Calculated Grain Boundary,Interant electrons,Calculated Yield Strength,Length of Basal phase,Habit Plane,Fraction of basal phase,Yang omega,Distribution of precipitation,Grain Size,Yield_Strength,Tensile_Strength （UTS）
MagpieData avg_dev MeltingT,,,,,,,,,,,,,,,,,,,,
34.914719,8.7,1.506678,0.000000,10.00,5.0,0.078602,8.7,50.0,10.0,69.060914,13,162.227111,50.0,3,4.9,2.068650,3,39.30,386,464
36.714576,8.6,1.506549,0.000000,10.00,5.0,0.056914,8.6,50.0,10.0,62.002431,14,151.402710,50.0,3,6.7,2.262757,3,40.10,442,493
38.606531,20.0,1.506467,0.000000,10.00,10.0,0.040407,20.0,50.0,10.0,57.869389,14,145.412261,50.0,3,7.4,2.227139,3,44.10,420,487
47.987890,8.3,1.509229,0.000000,10.00,5.0,0.080979,8.3,50.0,10.0,115.114434,14,222.043020,50.0,3,8.8,2.009091,3,14.50,448,520
31.450000,30.0,1.506259,0.001386,4.00,10.0,0.056204,30.0,0.0,0.0,24.683832,11,106.683007,0.0,7,0.0,2.082778,1,150.00,139,240
38.519115,30.0,1.507563,0.001342,12.00,10.0,0.056943,30.0,0.0,0.0,32.432694,11,122.309990,0.0,7,0.0,2.023153,1,90.00,154,220
53.260499,30.0,1.510281,0.001311,20.00,10.0,0.057791,30.0,0.0,0.0,50.296751,11,155.338573,0.0,7,0.0,1.902599,1,40.00,289,340
11.445493,10.0,1.501428,0.000551,0.97,5.0,0.010822,10.0,0.0,0.0,41.323512,11,86.600643,0.0,7,0.0,2.755676,1,52.60,244,330
43.818206,10.0,1.507114,0.000000,26.40,6.6,0.066328,10.0,60.0,10.0,52.857899,14,148.724167,60.0,3,10.0,3.005977,3,60.00,427,509


,Length of prismatic phase,Zr fraction,Fraction of prismatic phase,Width of prismatic phase,Diameter of prismatic phase,Diameter of basal phase,Width of basal phase,MagpieData range GSvolume_pa,Mixing enthalpy,Calculated Grain Boundary,MagpieData mean GSvolume_pa,Interant electrons,Calculated Yield Strength,Length of Basal phase,Habit Plane,Fraction of basal phase,Distribution of precipitation,Grain Size,Yield_Strength,Tensile_Strength （UTS）
Interant d electrons,,,,,,,,,,,,,,,,,,,,
6,8.7,0.000000,10.00,5.0,8.7,50.0,10.0,21.562414,0.617308,69.060914,23.054942,13,162.227111,50.0,3,4.9,3,39.30,386,464
7,8.6,0.000000,10.00,5.0,8.6,50.0,10.0,21.877414,0.594521,62.002431,23.056256,14,151.402710,50.0,3,6.7,3,40.10,442,493
7,20.0,0.000000,10.00,10.0,20.0,50.0,10.0,21.877414,0.620813,57.869389,23.055053,14,145.412261,50.0,3,7.4,3,44.10,420,487
7,8.3,0.000000,10.00,5.0,8.3,50.0,10.0,21.877414,0.810253,115.114434,23.138681,14,222.043020,50.0,3,8.8,3,14.50,448,520
4,30.0,0.001386,4.00,10.0,30.0,0.0,0.0,9.475000,0.450735,24.683832,23.080505,11,106.683007,0.0,7,0.0,1,150.00,139,240
4,30.0,0.001342,12.00,10.0,30.0,0.0,0.0,9.475000,0.551336,32.432694,23.121653,11,122.309990,0.0,7,0.0,1,90.00,154,220
4,30.0,0.001311,20.00,10.0,30.0,0.0,0.0,9.475000,0.756347,50.296751,23.207403,11,155.338573,0.0,7,0.0,1,40.00,289,340
4,10.0,0.000551,0.97,5.0,10.0,0.0,0.0,18.405000,0.207469,41.323512,22.917290,11,86.600643,0.0,7,0.0,1,52.60,244,330
7,10.0,0.000000,26.40,6.6,10.0,60.0,10.0,21.877414,0.522342,52.857899,23.043240,14,148.724167,60.0,3,10.0,3,60.00,427,509


(53, 20)


Exception ignored in: <function ZipFile.__del__ at 0x0000019BD082C7C0>
Traceback (most recent call last):
  File "D:\Anaconda\Lib\zipfile\__init__.py", line 1940, in __del__
    self.close()
  File "D:\Anaconda\Lib\zipfile\__init__.py", line 1957, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file
Exception ignored in: <function ZipFile.__del__ at 0x0000019BD082C7C0>
Traceback (most recent call last):
  File "D:\Anaconda\Lib\zipfile\__init__.py", line 1940, in __del__
    self.close()
  File "D:\Anaconda\Lib\zipfile\__init__.py", line 1957, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


🔸 **样本 0 在 Fraction of basal phase 下的预测范围**: 384.40 ~ 387.30 (差值 = 2.90)


IndexError: At least one sheet must be visible

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import joblib

# 定义文件夹路径
folder_path_qf = "final_model/qf/rf"
# 权重定义
weights_qf = { 'rf': 0.2}
sheet_name='qf_21'
# 加载测试集
df_qf = pd.read_excel('20240607FULL_final.xlsx', index_col=0,sheet_name=sheet_name)
X_test_qf = df_qf.iloc[:, :-2]
y_test_qf = df_qf['Yield_Strength']

# 选择一个样本
sample_index = 6  # 根据需要更改样本索引
sample = X_test_qf.iloc[sample_index].copy()
# print(X_test_qf.iloc[sample_index,-5:],df_qf.iloc[sample_index]['Yield_Strength'])
# 特征及其变化范围
feature_ranges = {
    # 'Calculated Yield Strength': np.arange(50, 301, 1),  # 已知
    'Width of basal phase': np.arange(0.5, 20, 0.2),
    'Diameter of prismatic phase': np.arange(0, 31, 1),
}
# 创建Excel文件以保存结果
with pd.ExcelWriter('qf_multiple_single_variable_analysis.xlsx', engine='openpyxl') as writer:
    for feature, ranges in feature_ranges.items():
        predictions_changes = []
        for value in ranges:
            modified_sample = sample.copy()
            modified_sample[feature] = value
            modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_qf.columns)

            prediction_for_this_value = np.zeros(1)
            # 遍历每个模型进行预测
            for model_name, weight in weights_qf.items():
                for i in range(0,5):
                    model_path = f'{folder_path_qf}/{model_name}_{i}.pkl'
                    model = joblib.load(model_path)
                    prediction_for_this_value += model.predict(modified_sample_df) * weight

            predictions_changes.append((value, prediction_for_this_value[0]))

        # 将结果转换为 DataFrame 并保存到 Excel的不同sheet中
        results_df = pd.DataFrame(predictions_changes, columns=[feature, 'Predicted Yield Strength'])
        results_df.to_excel(writer, sheet_name=feature, index=False)

print("Multi-feature single variable analysis completed and results saved to Excel.")



# 定义文件夹路径
folder_path_kl = "final_model/kl/rf"

# 权重定义
weights_kl = {'rf':0.2}
sheet_name='kl_21'

# 加载测试集
df_kl = pd.read_excel('20240607FULL_final.xlsx', index_col=0,sheet_name=sheet_name)
X_test_kl = df_kl.iloc[:, :-2]
y_test_kl = df_kl['Tensile_Strength （UTS）']

# 选择一个样本

# 创建Excel文件以保存结果
with pd.ExcelWriter('kl_multiple_single_variable_analysis.xlsx', engine='openpyxl') as writer:
    for feature, ranges in feature_ranges.items():
        predictions_changes = []
        for value in ranges:
            modified_sample = sample.copy()
            modified_sample[feature] = value
            modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_kl.columns)
            prediction_for_this_value = np.zeros(1)
            # 遍历每个模型进行预测
            for model_name, weight in weights_kl.items():
                for i in range(0,5):
                    model_path = f'{folder_path_kl}/{model_name}_{i}.pkl'
                    model = joblib.load(model_path)
                    prediction_for_this_value += model.predict(modified_sample_df) * weight

            predictions_changes.append((value, prediction_for_this_value[0]))

        # 将结果转换为 DataFrame 并保存到 Excel的不同sheet中
        results_df = pd.DataFrame(predictions_changes, columns=[feature, 'Predicted Tensile Strength'])
        results_df.to_excel(writer, sheet_name=feature, index=False)

print("Multi-feature single variable analysis for tensile strength completed and results saved to Excel.")


Multi-feature single variable analysis completed and results saved to Excel.


Exception ignored in: <function ZipFile.__del__ at 0x000002232C1957E0>
Traceback (most recent call last):
  File "F:\Anaconda\envs\new_env\lib\zipfile.py", line 1833, in __del__
    self.close()
  File "F:\Anaconda\envs\new_env\lib\zipfile.py", line 1850, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


Multi-feature single variable analysis for tensile strength completed and results saved to Excel.


In [0]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import joblib

# 定义文件夹路径
folder_path_qf = "qf_models"
# 权重定义
weights_qf = {'xgboost': 0.7 / 5, 'rf': 0.3 / 5}

# 加载测试集
df_qf = pd.read_excel('qf_models/train_set_new.xlsx', index_col=0)
X_test_qf = df_qf.drop(columns=['Precipitate Distribution',  'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_test_qf = df_qf['Yield_Strength']

# 选择样本索引
sample_indices = [50, 320, 470]  # 根据需要更改样本索引
print(X_test_qf.iloc[sample_indices,-5:],df_qf.iloc[sample_indices]['Yield_Strength'])

# 特征及其变化范围
feature_ranges = {
    'Calculated Yield Strength': np.arange(50, 301, 20),  # 已知
    'Grain Size': np.arange(5, 31, 1),  # 已知
    'Y fraction':np.arange(0.005,0.04,0.001),
    'Gd fraction':np.arange(0.005,0.04,0.001)
}

# 创建Excel文件以保存结果
with pd.ExcelWriter('qf_multiple_single_variable_analysis.xlsx', engine='openpyxl') as writer:
    for sample_index in sample_indices:
        sample = X_test_qf.iloc[sample_index].copy()
        for feature, ranges in feature_ranges.items():
            predictions_changes = []
            for value in ranges:
                modified_sample = sample.copy()
                modified_sample[feature] = value
                modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_qf.columns)

                prediction_for_this_value = np.zeros(1)
                # 遍历每个模型进行预测
                for model_name, weight in weights_qf.items():
                    for i in range(1, 6):
                        model_path = f'{folder_path_qf}/{model_name}{i}_new.pkl'
                        model = joblib.load(model_path)
                        prediction_for_this_value += model.predict(modified_sample_df) * weight

                predictions_changes.append((value, prediction_for_this_value[0]))

            # 将结果转换为 DataFrame 并保存到 Excel的不同sheet中
            sheet_name = f'sample_{sample_index}_{feature}'
            results_df = pd.DataFrame(predictions_changes, columns=[feature, 'Predicted Yield Strength'])
            results_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Multi-feature single variable analysis completed and results saved to Excel.")


In [2]:
import pandas as pd
sheet_name = 'qf_21'
df_qf = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name=sheet_name)
print(df_qf.columns)

Index(['Length of prismatic phase', 'mean AtomicRadius', 'Zr fraction',
       'Fraction of prismatic phase', 'Width of prismatic phase',
       'frac f valence electrons', 'Diameter of prismatic phase',
       'Diameter of basal phase', 'Width of basal phase',
       'Calculated Grain Boundary', 'Interant electrons',
       'Calculated Yield Strength', 'Length of Basal phase', 'Habit Plane',
       'Fraction of basal phase', 'Yang omega',
       'Distribution of precipitation', 'Grain Size', '屈服强度', '抗拉强度 （UTS）'],
      dtype='object')


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import joblib

# 定义文件夹路径
folder_path_qf = "final_model/qf/rf"
# 权重定义
weights_qf = {'rf': 0.2}

problem_name = 'qf'
model_name = 'rf'
sheet_name = 'qf_21'
df_qf = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name=sheet_name)

# 加载测试集
X_test_qf = df_qf.iloc[:, :-2]
y_test_qf = df_qf['Yield_Strength']

# 选择一个样本
# sample_indices = [4, 17, 36]  # 根据需要更改样本索引
sample_indices = np.arange(X_test_qf.shap[0])

print(X_test_qf.iloc[sample_indices,-5:],df_qf.iloc[sample_indices]['Yield_Strength'])

# 特征及其变化范围
feature_ranges = {
    # 'Diameter of basal phase': np.arange(1, 100, 1), # 已知
    # 'Width of basal phase': np.arange(0.5, 20, 0.2), # 已知
    # 'Diameter of prismatic phase': np.arange(0, 31, 0.5), # 已知
    'Fraction of basal phase':np.arange(0,21,0.5),
    'Fraction of prismatic phase':np.arange(0,21,0.5)
}
# 创建Excel文件以保存结果
with pd.ExcelWriter('AAA_qf_multiple_single_variable_analysis_fraction.xlsx', engine='openpyxl') as writer:
    for sample_index in sample_indices:
        sample = X_test_qf.iloc[sample_index].copy()
        for feature, ranges in feature_ranges.items():
            predictions_changes = []
            for value in ranges:
                modified_sample = sample.copy()
                modified_sample[feature] = value
                modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_qf.columns)

                prediction_for_this_value = np.zeros(1)
                # 遍历每个模型进行预测
                for model_name, weight in weights_qf.items():
                    for i in range(0, 5):
                        model_path = f'{folder_path_qf}/{model_name}_{i}.pkl'
                        model = joblib.load(model_path)
                        prediction_for_this_value += model.predict(modified_sample_df) * weight

                predictions_changes.append((value, prediction_for_this_value[0]))

            # 将结果转换为 DataFrame 并保存到 Excel的不同sheet中
            sheet_name = f'sample_{sample_index}_{feature}'
            results_df = pd.DataFrame(predictions_changes, columns=[feature, 'Predicted Yield Strength'])
            results_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Multi-feature single variable analysis completed and results saved to Excel.")


import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import joblib

# 定义文件夹路径
folder_path_kl = "final_model/kl/rf"
# 权重定义
weights_kl = {'rf': 0.2}

problem_name = 'kl'
model_name = 'rf'
sheet_name = 'kl_21'
df_kl = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name=sheet_name)
print(df_kl.columns)
# 加载测试集
X_test_kl = df_kl.iloc[:, :-2]
y_test_kl = df_kl['Tensile_Strength （UTS）']

sample_indices = np.arange(X_test_qf.shap[0])

print(X_test_qf.iloc[sample_indices,-5:],df_qf.iloc[sample_indices]['Yield_Strength'])

# 特征及其变化范围
feature_ranges = {
    'Diameter of basal phase': np.arange(1, 100, 1), # 已知
    'Width of basal phase': np.arange(0.5, 20, 0.2), # 已知
    'Diameter of prismatic phase': np.arange(0, 31, 0.5), # 已知
}

# 创建Excel文件以保存结果
with pd.ExcelWriter('kl_multiple_single_variable_analysis_fraction.xlsx', engine='openpyxl') as writer:
    for sample_index in sample_indices:
        sample = X_test_kl.iloc[sample_index].copy()
        for feature, ranges in feature_ranges.items():
            predictions_changes = []
            for value in ranges:
                modified_sample = sample.copy()
                modified_sample[feature] = value
                modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_kl.columns)

                prediction_for_this_value = np.zeros(1)
                # 遍历每个模型进行预测
                for model_name, weight in weights_kl.items():
                    for i in range(0, 5):
                        model_path = f'{folder_path_kl}/{model_name}_{i}.pkl'
                        # print(model_path)
                        model = joblib.load(model_path)
                        prediction_for_this_value += model.predict(modified_sample_df) * weight

                predictions_changes.append((value, prediction_for_this_value[0]))

            # 将结果转换为 DataFrame 并保存到 Excel的不同sheet中
            sheet_name = f'sample_{sample_index}_{feature}'
            results_df = pd.DataFrame(predictions_changes, columns=[feature, 'Predicted Tensile Strength'])
            results_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Multi-feature single variable analysis for tensile strength completed and results saved to Excel.")



In [9]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import joblib

# 定义文件夹路径
folder_path_qf = "final_model/qf/rf"
# 权重定义
weights_qf = {'rf': 0.2}

problem_name = 'qf'
model_name = 'rf'
sheet_name = 'qf_21'
df_qf = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name=sheet_name)

# 加载测试集
X_test_qf = df_qf.iloc[:, :-2]
y_test_qf = df_qf['Yield_Strength']

# 选择一个样本
# sample_indices = [4, 17, 36]  # 根据需要更改样本索引
sample_indices = [29,30]  # 根据需要更改样本索引

print(X_test_qf.iloc[sample_indices,-5:],df_qf.iloc[sample_indices]['Yield_Strength'])

# 特征及其变化范围
feature_ranges = {
    'Diameter of basal phase': np.arange(1, 100, 1), # 已知
    'Width of basal phase': np.arange(0.5, 20, 0.2), # 已知
    'Diameter of prismatic phase': np.arange(0, 31, 0.5), # 已知
}
# 创建Excel文件以保存结果
with pd.ExcelWriter('qf_multiple_single_variable_analysis.xlsx', engine='openpyxl') as writer:
    for sample_index in sample_indices:
        sample = X_test_qf.iloc[sample_index].copy()
        for feature, ranges in feature_ranges.items():
            predictions_changes = []
            for value in ranges:
                modified_sample = sample.copy()
                modified_sample[feature] = value
                modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_qf.columns)

                prediction_for_this_value = np.zeros(1)
                # 遍历每个模型进行预测
                for model_name, weight in weights_qf.items():
                    for i in range(0, 5):
                        model_path = f'{folder_path_qf}/{model_name}_{i}.pkl'
                        model = joblib.load(model_path)
                        prediction_for_this_value += model.predict(modified_sample_df) * weight

                predictions_changes.append((value, prediction_for_this_value[0]))

            # 将结果转换为 DataFrame 并保存到 Excel的不同sheet中
            sheet_name = f'sample_{sample_index}_{feature}'
            results_df = pd.DataFrame(predictions_changes, columns=[feature, 'Predicted Yield Strength'])
            results_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Multi-feature single variable analysis completed and results saved to Excel.")


                             Habit Plane  Fraction of basal phase  Yang omega  \
MagpieData avg_dev MeltingT                                                     
39.624073                              3                      7.0    2.020819   
42.812468                              3                      8.0    2.096332   

                             Distribution of precipitation  Grain Size  
MagpieData avg_dev MeltingT                                             
39.624073                                                1         1.5  
42.812468                                                3       138.0   MagpieData avg_dev MeltingT
39.624073    285
42.812468    194
Name: 屈服强度, dtype: int64


F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Multi-feature single variable analysis completed and results saved to Excel.


F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import joblib

# 定义文件夹路径
folder_path_kl = "final_model/kl/rf"
# 权重定义
weights_kl = {'rf': 0.2}

problem_name = 'kl'
model_name = 'rf'
sheet_name = 'kl_21'
df_kl = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name=sheet_name)
print(df_kl.columns)
# 加载测试集
X_test_kl = df_kl.iloc[:, :-2]
y_test_kl = df_kl['Tensile_Strength （UTS）']

# 选择一个样本
sample_indices = [29,30]  # 根据需要更改样本索引
print(X_test_kl.iloc[sample_indices,-5:],df_kl.iloc[sample_indices]['Tensile_Strength （UTS）'])

# 特征及其变化范围
feature_ranges = {
    # 'Diameter of basal phase': np.arange(0, 100, 5), # 已知
    # 'Width of basal phase': np.arange(0.5, 20, 0.5), # 已知
    # 'Diameter of prismatic phase': np.arange(0, 31, 1), # 已知

}
# 创建Excel文件以保存结果
with pd.ExcelWriter('kl_multiple_single_variable_analysis.xlsx', engine='openpyxl') as writer:
    for sample_index in sample_indices:
        sample = X_test_kl.iloc[sample_index].copy()
        for feature, ranges in feature_ranges.items():
            predictions_changes = []
            for value in ranges:
                modified_sample = sample.copy()
                modified_sample[feature] = value
                modified_sample_df = pd.DataFrame([modified_sample], columns=X_test_kl.columns)

                prediction_for_this_value = np.zeros(1)
                # 遍历每个模型进行预测
                for model_name, weight in weights_kl.items():
                    for i in range(0, 5):
                        model_path = f'{folder_path_kl}/{model_name}_{i}.pkl'
                        # print(model_path)
                        model = joblib.load(model_path)
                        prediction_for_this_value += model.predict(modified_sample_df) * weight

                predictions_changes.append((value, prediction_for_this_value[0]))

            # 将结果转换为 DataFrame 并保存到 Excel的不同sheet中
            sheet_name = f'sample_{sample_index}_{feature}'
            results_df = pd.DataFrame(predictions_changes, columns=[feature, 'Predicted Tensile Strength'])
            results_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Multi-feature single variable analysis for tensile strength completed and results saved to Excel.")



Index(['Length of prismatic phase', 'Zr fraction',
       'Fraction of prismatic phase', 'Width of prismatic phase',
       'Diameter of prismatic phase', 'Diameter of basal phase',
       'Width of basal phase', 'MagpieData range GSvolume_pa',
       'Mixing enthalpy', 'Calculated Grain Boundary',
       'MagpieData mean GSvolume_pa', 'Interant electrons',
       'Calculated Yield Strength', 'Length of Basal phase', 'Habit Plane',
       'Fraction of basal phase', 'Distribution of precipitation',
       'Grain Size', '屈服强度', '抗拉强度 （UTS）'],
      dtype='object')
                      Length of Basal phase  Habit Plane  \
Interant d electrons                                       
5                                      26.0            3   
4                                     100.0            3   

                      Fraction of basal phase  Distribution of precipitation  \
Interant d electrons                                                           
5                             

F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Multi-feature single variable analysis for tensile strength completed and results saved to Excel.


F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
